# Executable sentiment and topic classification

This notebook is an execution interface. All reusable logic lives in `src/`.

## Goal

Train and evaluate two end-to-end LSTM classifiers for product reviews: sentiment and product topic.

## Setup

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Run the notebook from inside the repository.')

sys.path.insert(0, str(ROOT))

from src.sentiment_classifier import (
    SentimentPipelineConfig,
    run_sentiment_pipeline,
)
from src.topic_classifier import TopicPipelineConfig, run_topic_pipeline

print('Environment configured.')

Environment configured.


## Steps

### 1. Configure both classifiers

The number of epochs can be controlled with the `PIPELINE_EPOCHS` environment variable.

In [2]:
epochs = int(os.getenv('PIPELINE_EPOCHS', '20'))

sentiment_config = SentimentPipelineConfig(
    dataset_path=ROOT / 'data' / 'sentiment_samples.csv',
    epochs=epochs,
    seed=42,
)
topic_config = TopicPipelineConfig(
    dataset_path=ROOT / 'data' / 'topic_samples.csv',
    epochs=epochs,
    seed=42,
)

### 2. Train and evaluate sentiment

In [3]:
sentiment_result = run_sentiment_pipeline(sentiment_config)

sentiment_summary = {
    'dataset_size': sentiment_result.dataset_size,
    'train_size': sentiment_result.train_size,
    'test_size': sentiment_result.test_size,
    'labels': sentiment_result.labels,
    'epochs': epochs,
    'test_accuracy': round(sentiment_result.metrics['accuracy'], 4),
    'test_loss': round(sentiment_result.metrics['loss'], 4),
    'confusion_matrix': sentiment_result.confusion_matrix,
}

sentiment_summary

{'dataset_size': 90,
 'train_size': 72,
 'test_size': 18,
 'labels': ['negative', 'neutral', 'positive'],
 'epochs': 20,
 'test_accuracy': 1.0,
 'test_loss': 0.0258,
 'confusion_matrix': [[6, 0, 0], [0, 6, 0], [0, 0, 6]]}

In [4]:
sentiment_result.predictions[:6]

[{'text': 'ordinary purchase with standard delivery',
  'expected': 'neutral',
  'predicted': 'neutral'},
 {'text': 'good purchase that i would recommend',
  'expected': 'positive',
  'predicted': 'positive'},
 {'text': 'ordinary model with standard specifications',
  'expected': 'neutral',
  'predicted': 'neutral'},
 {'text': 'terrible service and bad support',
  'expected': 'negative',
  'predicted': 'negative'},
 {'text': 'standard product available in black',
  'expected': 'neutral',
  'predicted': 'neutral'},
 {'text': 'excellent quality and good reliability',
  'expected': 'positive',
  'predicted': 'positive'}]

### 3. Train and evaluate topics

In [5]:
topic_result = run_topic_pipeline(topic_config)

topic_summary = {
    'dataset_size': topic_result.dataset_size,
    'train_size': topic_result.train_size,
    'test_size': topic_result.test_size,
    'labels': topic_result.labels,
    'epochs': epochs,
    'test_accuracy': round(topic_result.metrics['accuracy'], 4),
    'test_loss': round(topic_result.metrics['loss'], 4),
    'confusion_matrix': topic_result.confusion_matrix,
}

topic_summary

{'dataset_size': 80,
 'train_size': 64,
 'test_size': 16,
 'labels': ['refrigerator', 'smartphone', 'television', 'washing_machine'],
 'epochs': 20,
 'test_accuracy': 0.6875,
 'test_loss': 0.9746,
 'confusion_matrix': [[2, 1, 0, 1], [0, 2, 0, 2], [0, 0, 3, 1], [0, 0, 0, 4]]}

In [6]:
topic_result.predictions[:6]

[{'text': 'the fingerprint reader stopped recognizing me',
  'expected': 'smartphone',
  'predicted': 'washing_machine'},
 {'text': 'the television has sound but no picture',
  'expected': 'television',
  'predicted': 'television'},
 {'text': 'the refrigerator water dispenser stopped working',
  'expected': 'refrigerator',
  'predicted': 'refrigerator'},
 {'text': 'the refrigerator takes too long to reach temperature',
  'expected': 'refrigerator',
  'predicted': 'smartphone'},
 {'text': 'the touchscreen does not respond to commands',
  'expected': 'smartphone',
  'predicted': 'washing_machine'},
 {'text': 'the washing machine does not complete the spin cycle',
  'expected': 'washing_machine',
  'predicted': 'washing_machine'}]

## Checks

In [7]:
assert set(sentiment_result.labels) == {'negative', 'neutral', 'positive'}
assert set(topic_result.labels) == {
    'smartphone', 'television', 'refrigerator', 'washing_machine'
}
assert sentiment_result.metrics['accuracy'] >= 0.50, sentiment_result.metrics
assert topic_result.metrics['accuracy'] >= 0.50, topic_result.metrics
assert len(sentiment_result.predictions) == sentiment_result.test_size
assert len(topic_result.predictions) == topic_result.test_size

{
    'status': 'ok',
    'sentiment_accuracy': round(sentiment_result.metrics['accuracy'], 4),
    'topic_accuracy': round(topic_result.metrics['accuracy'], 4),
}

{'status': 'ok', 'sentiment_accuracy': 1.0, 'topic_accuracy': 0.6875}

## Next Steps

Replace the synthetic datasets with real, labeled, PII-free reviews. Preserve the `text,label` schema to reuse both pipelines.